# Initial EDA on marathos

## The uploaded CSV will be read into a spark dataframe and datatypes inferred

In [0]:
df = spark.read.csv(
    "/Volumes/marathos/default/raw/TWO_CENTURIES_OF_UM_RACES.csv",
    header=True,
    inferSchema=True
)

## The scope of the dataframe
- Number of records
- Number of columns

In [0]:
print(f"Number of rows: {df.count()}")
print(f"Number of columns: {len(df.columns)}")

## The names of the columns and their inferred values

In [0]:
df.printSchema()

## Some descriptive statistics on the columns that were numerical datatypes

In [0]:
df.select(
    [
        column
        for column, type_ in df.dtypes
        if type_ in ("int", "double")
    ]
).summary().display()

## Number of nulls in each column

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items()]

## Number of unique Event names in the dataset

In [0]:
df.select("Event name").distinct().count()

## The exact ages of the athletes can not be gathered directly, but still be inferred based on the relationship between each records columns - the year of the event subtracted by year of birth of the athlete

In [0]:
from pyspark.sql import functions as F

calculate_age = df.withColumn("athlete_age", F.col("Year of event") - F.col("Athlete year of birth"))
display(calculate_age.groupBy("athlete_age").count().orderBy("athlete_age"))

## Counting each country and presenting them by country

In [0]:
count_country = df.groupBy("Athlete country").count().orderBy("Athlete country")
display(count_country)

## Counting the number of finishers and presenting them 

In [0]:
count_finishers = df.groupBy("Event number of finishers").count().orderBy("Event number of finishers").limit(10)
display(count_finishers)

# Further deep dive on each column


## Distribution of event years seem reasonable as ultramarathons and data have become more popular with time

In [0]:
count_event_years = df.groupBy("Year of event").count().orderBy("Year of event")
display(count_event_years)

## Year of event summary
- Integer reasonable for this column
- 0 nulls
- Expected range of years from looking at the descriptive statistics
- Should not need further cleaning based on this